# Reasons Analysis

Full pipeline for analysing ICF (Interval Constraint Function) reasons and anti-reasons
from MCR experiment checkpoints:

1. Load checkpoint from directory
2. Extract test samples and compute sigmas
3. Calculate costs (parallel, incremental cache)
4. Compute robustness per ICF and per sample
5. Visualise cost distributions, robustness, and ICF constraints

**Robustness formula (IEEE CAI 2026)**:
$$r(C, x) = 1 - \frac{\max_{\text{ICF} \in AR_{C,y}} \text{cost}_x(\text{ICF})}{|\text{features}|}$$

In [ ]:
from pathlib import Path
from etl.loader import etl_from_dir, list_checkpoints

RESULTS_DIR = Path("results")

# List available checkpoints
checkpoints = list_checkpoints(RESULTS_DIR)
print(f"Found {len(checkpoints)} checkpoints:")
for i, cp in enumerate(checkpoints):
    print(f"  [{i}] {cp.relative_to(RESULTS_DIR)}")

# Select checkpoint (edit index or set CHECKPOINT_DIR directly)
CHECKPOINT_DIR = checkpoints[0] if checkpoints else None
print(f"\nUsing: {CHECKPOINT_DIR}")

In [ ]:
# Load checkpoint
db = etl_from_dir(CHECKPOINT_DIR, verbose=True)
dataset_name = db['_dataset_name']

In [ ]:
# Extract test samples and compute split-Gaussian sigmas
from cost_function import cal_sigmas
from etl.reasons_analysis import extract_test_samples

tests_sample, X_test, test_ids, feature_names = extract_test_samples(db)

X_train = db["data"]["TRAINING_SET"].get("value_json", db["data"]["TRAINING_SET"])["X_train"]
sigmas_all = cal_sigmas(X_train, X_test, feature_names, test_ids=test_ids)

print(f"Extracted {len(test_ids)} test samples")
print(f"Features: {len(feature_names)}")

In [ ]:
# Calculate costs for all ICFs (parallel, incremental)
from cost_function import cost_function
from redis_helpers.icf import bitmap_to_icf
import pandas as pd

reason_types = ['reasons', 'non_reasons', 'anti_reasons']

print("Calculating costs for all reason types...")
print("=" * 80)

cost_records = []
eu_raw = db["data"]['EU']
eu = eu_raw.get("value_json", eu_raw) if isinstance(eu_raw, dict) and "value_json" in eu_raw else eu_raw

# Expected bitmap length derived from EU
expected_bitmap_len = sum(len(v) - 1 for v in eu.values()) + len(eu)
print(f"Expected bitmap length: {expected_bitmap_len}")

for reason_type in reason_types:
    if reason_type not in db or not db[reason_type]:
        print(f"  {reason_type}: not found or empty, skipping")
        continue
    bitmaps = [k for k in db[reason_type].keys() if set(k) <= {'0', '1'}]
    print(f"  {reason_type}: {len(bitmaps)} ICFs")
    for bitmap_string in bitmaps:
        try:
            icf = bitmap_to_icf(bitmap_string, eu)
        except Exception as e:
            print(f"    skip bitmap (len={len(bitmap_string)}): {e}")
            continue
        for sample_id in test_ids:
            sample_data = tests_sample[sample_id]
            sample_data.setdefault(reason_type, {})
            cost = cost_function(
                sample=sample_data['features'],
                sigmas=sigmas_all[sample_id],
                icf=icf
            )
            sample_data[reason_type][bitmap_string] = {'icf': icf, 'cost': cost}
            cost_records.append({
                'reason_type': reason_type,
                'bitmap': bitmap_string,
                'sample_id': sample_id,
                'cost': cost,
                'icf': icf,
            })

cost_df = pd.DataFrame(cost_records)
print(f"\nTotal costs calculated: {len(cost_df)}")
print(cost_df.groupby('reason_type')['cost'].describe())

## Robustness Calculation

Per-sample robustness using anti-reasons only (as per the IEEE CAI 2026 paper).

In [ ]:
from etl.reasons_analysis import (
    calculate_robustness_per_bitmap,
    calculate_all_samples_robustness,
    print_robustness_statistics
)

num_features = len(feature_names)

print(f"{'='*80}")
print(f"  ROBUSTNESS CALCULATION (Anti-Reasons Only)")
print(f"{'='*80}\n")
print(f"  Number of features: {num_features}")
print(f"  Formula: r(C,x) = 1 - max{{ICF ∈ AR{{C,y}}}} cost_x(ICF) / |features|")

anti_reasons_df = cost_df[cost_df['reason_type'] == 'anti_reasons'].copy()

if len(anti_reasons_df) == 0:
    print("\nWARNING: No anti_reasons found in cost data.")
else:
    bitmap_robustness = calculate_robustness_per_bitmap(anti_reasons_df, num_features=num_features)

    print(f"\n  Total anti-reason ICFs: {len(bitmap_robustness)}")
    print(f"  Max cost:      {bitmap_robustness['max_cost'].max():.6f}")
    print(f"  Mean max cost: {bitmap_robustness['max_cost'].mean():.6f}")
    print(f"  Min max cost:  {bitmap_robustness['max_cost'].min():.6f}")

    sample_robustness_df = calculate_all_samples_robustness(
        cost_df=cost_df,
        num_features=num_features,
        tests_sample=tests_sample,
        test_ids=test_ids,
        verbose=True
    )

    print_robustness_statistics(sample_robustness_df)

    Path('results').mkdir(exist_ok=True)
    sample_robustness_df.to_csv('results/sample_robustness.csv', index=False)
    print(f"\nResults saved to: results/sample_robustness.csv")

In [ ]:
# Robustness distribution visualisations (plotly)
from etl.reasons_analysis import create_robustness_visualizations

if 'sample_robustness_df' in locals() and len(sample_robustness_df) > 0:
    fig_main, fig_quartiles = create_robustness_visualizations(
        sample_robustness_df=sample_robustness_df,
        dataset_name=dataset_name
    )
    if fig_main is not None:
        fig_main.show()
    if fig_quartiles is not None:
        fig_quartiles.show()
else:
    print("sample_robustness_df not available. Run previous cell first.")

In [ ]:
# Feature-level cost distribution visualisations
# Skipped for large datasets to avoid memory issues
print("Cost distribution visualization skipped (use sample-level analysis above).")

In [ ]:
# Select a sample for detailed ICF visualisation
sample_id = test_ids[0]
print(f"Analysing sample: {sample_id}")
print(f"Features: {feature_names[:10]}{'...' if len(feature_names) > 10 else ''}")

In [ ]:
# Visualise per-feature ICF constraints as bar charts (tabular data)
from etl.reasons_analysis import visualize_sample_with_icf

# Reason ICF
fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='reasons')
if fig is not None:
    fig.show()

In [ ]:
# Anti-Reason ICF
fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='anti_reasons')
if fig is not None:
    fig.show()

In [ ]:
# Reason vs Anti-Reason comparison
from etl.reasons_analysis import visualize_sample_comparison

fig = visualize_sample_comparison(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()

In [ ]:
# Smooth corridor comparison
from etl.reasons_analysis import visualize_sample_comparison_smooth

fig = visualize_sample_comparison_smooth(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()

In [ ]:
# Anti-reason corridor coloured by constraint satisfaction
from etl.reasons_analysis import visualize_anti_reason_corridor

fig = visualize_anti_reason_corridor(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()

In [ ]:
# Robustness per ICF (bitmap-level)
if 'anti_reasons_df' in locals() and len(anti_reasons_df) > 0:
    print("="*80)
    print("ROBUSTNESS PER ICF (Anti-Reasons Only)")
    print("="*80)
    print(f"\nTotal anti-reason ICFs: {len(bitmap_robustness)}")
    print("\nTop 10 by max_cost (hardest anti-reasons):")
    print(bitmap_robustness[['max_cost', 'robustness', 'mean_cost', 'n_samples']].head(10).to_string())

    print("\nBottom 5 robustness (most vulnerable):")
    ar_lowest = bitmap_robustness.nsmallest(5, 'robustness')
    print(ar_lowest[['max_cost', 'robustness', 'mean_cost', 'n_samples']].to_string())

    bitmap_robustness.to_csv('results/anti_reasons_robustness.csv', index=False)
    print("\nSaved to: results/anti_reasons_robustness.csv")